
# <font color="green">Literal (immediate) values</font>

## Problem

* Write two functions __in assembly__:
```
long imm() { return 1234567; }
double fimm() { return 1.234; }
```
  * `imm` returns the `long` value `1234567`. Since this does not fit in a single 16-bit immediate, you will need a `mov`/`movz` followed by one or more `movk`s.
  * `fimm` returns the `double` value `1.234`. Since this is not a "simple" number, build its 64-bit representation on an integer register first, then move it into a floating-point register with `fmov`.
* Fill in the skeleton `literal.s` (after each `// ------- write your answer here -------`) with instructions.
* The checker `check_literal.c` verifies both return values. If you see `OK`s and no errors, you are done.

## Hints

* Using literal (immediate) values in arbitrary expressions (e.g., `x + 1234567` or `x * 3.141592`) is trivial in any high-level language, but not in machine code.
* Machine languages restrict using such values directly in instructions, because the number of bits per instruction is limited (32 bits in ARM64); there is no room to encode an arbitrary 32-bit, let alone 64-bit, number.
* In ARM64:
  * a `mov`/`movz` (move-and-zero) instruction can set one of the four 16-bit words of an integer register to a specified 16-bit value, and zeros the remaining three words;
  * a `movk` (move-and-keep) instruction can set one of the four 16-bit words to a specified 16-bit value, and leaves the remaining three words unchanged;
  * by combining a `mov` with up to three `movk`s, you can set an arbitrary 64-bit value into a register;
  * `fmov` can set a floating-point register to certain "simple" numbers --- numbers whose exponent and mantissa fit in a few bits; in a quick investigation, `fmov` can take numbers of the form $\pm 1.xxxx \times 2^{(yyy-3)}$ (3-bit exponent, 4-bit mantissa, positive and negative);
  * each non-simple number is first built as its bit representation on an integer register with `mov`/`movk`, and then moved to a floating-point register with `fmov`.
* The *Observe* cells below contain `imm` and `fimm` (returning `1234567` and `1.234`). Compile them and look at how the constants are loaded. Try changing the immediate values --- e.g. a small int (`5`), a "simple" floating-point value (`1.5`), and a "non-simple" one (`1.234`) --- and compare the generated code.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=literal.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* Literal values: observe how constants are loaded.
   Try changing 1234567 and 1.234, e.g. a small int (5), a "simple" double (1.5),
   and a "non-simple" double (1.234), and compare the generated code. */
long imm()   { return 1234567; }
double fimm() { return 1.234; }

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `literal.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ literal.s
	.arch armv8-a
	.file	"literal.c"
	.text
	.align	2
	.p2align 4,,11
	.global	imm
	.type	imm, %function
imm:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	// return the long value 1234567
	.cfi_endproc
.LFE0:
	.size	imm, .-imm

	.align	2
	.p2align 4,,11
	.global	fimm
	.type	fimm, %function
fimm:
.LFB1:
	.cfi_startproc
	// ------- write your answer here -------
	// return the double value 1.234
	.cfi_endproc
.LFE1:
	.size	fimm, .-fimm
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `literal` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_literal.c
#include <stdio.h>
#include <math.h>
long imm(void);
double fimm(void);

int main(void) {
  long i = imm();
  double f = fimm();
  int ok = 1;
  if (i == 1234567L) {
    printf("OK imm %ld\n", i);
  } else {
    printf("NG imm %ld (expected 1234567)\n", i);
    ok = 0;
  }
  if (fabs(f - 1.234) <= 1e-9) {
    printf("OK fimm %f\n", f);
  } else {
    printf("NG fimm %f (expected 1.234)\n", f);
    ok = 0;
  }
  return ok ? 0 : 1;
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `literal.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_literal -O3 check_literal.c literal.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_literal


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_literal -O0 -g check_literal.c literal.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_literal
(gdb) break literal
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=literal.md answer_file=literal.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.